In [ ]:
from pypdf import PdfReader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Chunking
pdf_path = r"D:\Amr\Portfolio\All time project\ready to publish\LLM\Project 2 RAG\machine learning journal.pdf"

reader = PdfReader(pdf_path)
full_text = "\n".join(page.extract_text() or "" for page in reader.pages)

splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
chunks = [c.page_content for c in splitter.create_documents([full_text])]
print(f"{len(chunks)} chunks created")

339 chunks created


In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

# Embedding
embedder = SentenceTransformer("BAAI/bge-small-en-v1.5")  # small, fast, fully local
chunk_embeddings = embedder.encode(chunks, normalize_embeddings=True)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [ ]:
import faiss

# Vector store
dim = chunk_embeddings.shape[1]
index = faiss.IndexFlatIP(dim)  # inner product == cosine similarity (embeddings are normalized)
index.add(np.array(chunk_embeddings, dtype="float32"))

In [ ]:
# Retrieval
def retrieve(query, k=5):
    q_emb = embedder.encode([query], normalize_embeddings=True)
    scores, idxs = index.search(np.array(q_emb, dtype="float32"), k)
    return [(chunks[i], float(s)) for i, s in zip(idxs[0], scores[0])]

In [ ]:
from sentence_transformers import CrossEncoder

# Re-ranking
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

def rerank(query, candidates, top_n=3):
    pairs = [(query, text) for text, _ in candidates]
    scores = reranker.predict(pairs)
    reranked = sorted(zip(candidates, scores), key=lambda x: x[1], reverse=True)
    return [c[0][0] for c in reranked[:top_n]]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

In [ ]:
import ollama

# Generation
def generate(query, context_chunks, model="llama3.1-local"):
    context = "\n\n".join(context_chunks)
    prompt = (
        "Answer the question using only the context below. "
        "If the answer isn't in the context, say you don't know.\n\n"
        f"Context:\n{context}\n\nQuestion: {query}\nAnswer:"
    )
    response = ollama.chat(model=model, messages=[{"role": "user", "content": prompt}])
    return response["message"]["content"]

In [ ]:
# Main pipeline
def rag(query, k=5, top_n=3):
    candidates = retrieve(query, k=k)
    top_chunks = rerank(query, candidates, top_n=top_n)
    return generate(query, top_chunks)

In [ ]:
# Example usage

# True example
answer = rag("What is the main topic?")
print('answer if explanation is in paper:')
print(answer)

# False example
answer = rag("What is computer vision?")
print('\nanswer if explanation is not in paper:')
print(answer)

answer if explanation is in paper:
The main topic appears to be the analysis of the relationship between density and capacitance, and the factors that influence it, likely in the context of materials science or electrochemistry.

answer if explanation is not in paper:
I don't know.
